# KMOS Emission Line Fitting — Top-5 per line group
# Shows maps + fitted spectra inline, no files written.

In [62]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from astropy.io import fits
from scipy.optimize import curve_fit
import glob, re, os, csv, warnings, pickle
from pathlib import Path

warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size': 9, 'figure.dpi': 110, 'savefig.bbox': 'tight'})

# ── Paths & constants ──────────────────────────────────────────────────────────
ROOT       = Path('.')
COMB_DIR   = ROOT / 'kmos_proc_server_comb'
Z_CSV      = ROOT / 'scripts' / 'target_redshifts.csv'
CACHE      = ROOT / 'kmos_emission_fits.pkl'
C_KMS      = 2.99792458e5

BAND_R = {'IZ': 3200, 'YJ': 3500, 'HK': 3000, 'H': 3500, 'K': 3500}
FIT_WIN    = 0.025   # ±µm around anchor line
VEL_RANGE  = 500.0   # km/s allowed offset
MAX_SIG_KMS= 400.0

# ── Load redshifts ─────────────────────────────────────────────────────────────
_Z = {}
with open(Z_CSV, newline='') as f:
    for row in csv.DictReader(f):
        if row.get('redshift', '').strip():
            key = (row['field'].strip(), row['arm_target_id'].strip())
            try:
                quality = float(row.get('quality') or 'nan')
            except (ValueError, TypeError):
                quality = np.nan
            _Z[key] = {'z': float(row['redshift']),
                       'type': row.get('redshift_type', '').strip(),
                       'quality': quality}
print(f'Redshifts loaded: {len(_Z)}')

Redshifts loaded: 104


In [63]:
# ── Cube utilities ─────────────────────────────────────────────────────────────

_CUBE_RE = re.compile(r'DATACUBE_COMBINED_(.+)_([^_]+)_(IZ|YJ|HK|H|K)\.fits$')

def wl_axis(hdr):
    cd = hdr.get('CDELT3', hdr.get('CD3_3', 1.0))
    return (np.arange(hdr['NAXIS3']) + 1 - hdr['CRPIX3']) * cd + hdr['CRVAL3']

def erode(finite, m=4):
    inv = ~finite; e = inv.copy()
    for s in range(1, m+1):
        e[s:] |= inv[:-s]; e[:-s] |= inv[s:]
    return ~e

def ap_spec(cube, cy, cx, box=5):
    h = box//2
    patch = cube[:, max(cy-h,0):cy+h+1, max(cx-h,0):cx+h+1].reshape(cube.shape[0], -1)
    v = erode(np.isfinite(patch))
    s = np.full(cube.shape[0], np.nan)
    ok = v.any(1)
    s[ok] = np.nansum(np.where(v[ok], patch[ok], np.nan), 1)
    return s

def peak_center(cube, wave, z, rest=0.656281, hw=0.004, maxd=6):
    lam = rest*(1+z); mask = np.abs(wave-lam)<hw
    cy0, cx0 = cube.shape[1]//2, cube.shape[2]//2
    if mask.sum() < 2: return cy0, cx0
    nb = np.nansum(cube[mask], 0)
    if not np.any(np.isfinite(nb)): return cy0, cx0
    p = np.unravel_index(np.nanargmax(nb), nb.shape)
    return p if (abs(p[0]-cy0)<=maxd and abs(p[1]-cx0)<=maxd) else (cy0, cx0)

def in_band(wave, rest, z, m=0.002):
    lam = rest*(1+z)
    return (wave.min()+m) < lam < (wave.max()-m)

def sig_inst(band, lam):
    return lam / BAND_R.get(band,3500) / (2*np.sqrt(2*np.log(2)))

def discover():
    out = []
    for p in sorted(COMB_DIR.rglob('DATACUBE_COMBINED_*.fits')):
        m = _CUBE_RE.match(p.name)
        if not m: continue
        field, tid, band = m.group(1), m.group(2), m.group(3)
        key = (field, tid)
        if key not in _Z:
            for k in _Z:
                if k[1]==tid: key=k; field=k[0]; break
        out.append(dict(path=p, field=field, tid=tid, band=band, z_info=_Z.get(key)))
    return out

CUBES = discover()
print(f'Found {len(CUBES)} cubes')

Found 84 cubes


In [64]:
# ── Gaussian models (all share same parameter layout) ─────────────────────────
# p = [cont_a, cont_b, z_sys, sigma_obs, amp1, (amp2)]

def _g(w, a, c, s): return a*np.exp(-0.5*((w-c)/max(s,1e-9))**2)

def m_ha_nii(w, ca, cb, z, s, aHa, aN83):
    return (ca*w+cb + _g(w,aHa,  0.656281*(1+z),s)
                    + _g(w,aN83, 0.658345*(1+z),s)
                    + _g(w,aN83/3,0.654805*(1+z),s))

def m_oiii(w, ca, cb, z, s, a5007):
    return (ca*w+cb + _g(w,a5007,  0.500824*(1+z),s)
                    + _g(w,a5007/3,0.495940*(1+z),s))

def m_sii(w, ca, cb, z, s, a16, a31):
    return (ca*w+cb + _g(w,a16,0.671644*(1+z),s)
                    + _g(w,a31,0.673082*(1+z),s))

def m_hb(w, ca, cb, z, s, aHb):
    return ca*w+cb + _g(w,aHb,0.486270*(1+z),s)

# group -> (fn, anchor_rest, n_amp_params)
GROUPS = {
    'Ha+[NII]': (m_ha_nii, 0.656281, 2),
    '[OIII]':   (m_oiii,   0.500824, 1),
    '[SII]':    (m_sii,    0.671644, 2),
    'Hbeta':    (m_hb,     0.486270, 1),
}
LCOLORS = {'Ha': '#e84545', '[NII]': '#f97306', '[OIII]': '#4fc3f7',
           '[SII]6716': '#66bb6a', '[SII]6731': '#2d8a40', 'Hbeta': '#ce93d8'}

print('Models ready')

Models ready


In [65]:
# ── Core fitter ────────────────────────────────────────────────────────────────

def fit_group(wave, spec, group, z, band, p0_over=None, tight_p0=None):
    """
    Returns dict(popt, snr, vel_kms, sigma_kms, z_fit, noise,
                 wave_win, spec_win, fluxes) or None.
    """
    fn, anchor, n_amp = GROUPS[group]
    if not in_band(wave, anchor, z):
        return None

    lam = anchor*(1+z)
    si  = sig_inst(band, lam)
    win = np.abs(wave-lam) < FIT_WIN
    n_par = 4 + n_amp

    ww = wave[win]; ss = spec[win]
    ok = np.isfinite(ss)
    if ok.sum() < n_par + 3: return None
    ww, ss = ww[ok], ss[ok]

    # Continuum from outer 12% of window
    ne = max(2, int(len(ww)*0.12))
    we = np.r_[ww[:ne], ww[-ne:]]; se = np.r_[ss[:ne], ss[-ne:]]
    try:   cf = np.polyfit(we, se, 1)
    except: cf = [0., float(np.nanmedian(ss))]

    resid_e = se - np.polyval(cf, we)
    noise = max(1.4826*np.nanmedian(np.abs(resid_e - np.nanmedian(resid_e))), 1e-20)
    amp0  = max(float(np.nanmax(ss - np.polyval(cf,ww))), noise*2)

    dz  = VEL_RANGE/C_KMS*(1+z)
    smax= (MAX_SIG_KMS/C_KMS)*lam + si

    if p0_over is not None:
        p0 = list(p0_over)
    else:
        p0 = [cf[0], cf[1], z, si*1.5] + [amp0]*n_amp

    cr = (np.nanmax(np.abs(ss))+1e-20)*2
    lo = [-cr, -cr*lam, z-dz, si]     + [0.]*n_amp
    hi = [ cr,  cr*lam, z+dz, smax]   + [amp0*30]*n_amp

    if tight_p0 is not None:
        lo[2] = max(lo[2], tight_p0[2]-dz*0.4)
        hi[2] = min(hi[2], tight_p0[2]+dz*0.4)
        lo[3] = max(lo[3], tight_p0[3]*0.4)
        hi[3] = min(hi[3], tight_p0[3]*2.5)

    p0 = np.clip(p0, lo, hi)
    try:
        popt, pcov = curve_fit(fn, ww, ss, p0=p0, bounds=(lo,hi),
                               method='trf', maxfev=3000,
                               sigma=np.full_like(ss,noise), absolute_sigma=True)
    except Exception: return None

    if not np.all(np.isfinite(np.diag(pcov))): return None
    if popt[3] < si*0.9: return None

    z_fit = popt[2]; s_obs = popt[3]
    s_int = np.sqrt(max(s_obs**2 - si**2, 0.))
    vel   = (z_fit-z)/(1+z)*C_KMS
    sigma_kms = C_KMS*s_int / (anchor*(1+z_fit))
    snr   = popt[4] / noise

    sq2pi = np.sqrt(2*np.pi)
    fluxes = {}
    if group == 'Ha+[NII]':
        fluxes = {'Ha': popt[4]*s_obs*sq2pi,
                  '[NII]6583': popt[5]*s_obs*sq2pi,
                  '[NII]6548': popt[5]/3*s_obs*sq2pi}
    elif group == '[OIII]':
        fluxes = {'[OIII]5007': popt[4]*s_obs*sq2pi,
                  '[OIII]4959': popt[4]/3*s_obs*sq2pi}
    elif group == '[SII]':
        fluxes = {'[SII]6716': popt[4]*s_obs*sq2pi,
                  '[SII]6731': popt[5]*s_obs*sq2pi}
        r = popt[4]/max(popt[5],1e-30)
        fluxes['ratio_6716_6731'] = float(np.clip(r,0.4,2.5))
    elif group == 'Hbeta':
        fluxes = {'Hbeta': popt[4]*s_obs*sq2pi}

    return dict(popt=popt, snr=snr, vel_kms=vel, sigma_kms=sigma_kms,
                z_fit=z_fit, s_obs=s_obs, noise=noise,
                wave_win=ww, spec_win=ss, fluxes=fluxes)


print('Fitter ready')

Fitter ready


In [66]:
# ── Phase 1: aperture SNR scan over all cubes (fast, no maps yet) ─────────────

FORCE = False  # set True to force a fresh scan

def _cache_valid(d):
    """Check that cached dict has the structure this notebook expects."""
    if not d: return False
    sample = next(iter(d.values()))
    return 'snrs' in sample and 'wave' in sample and 'spec' in sample

loaded = False
if CACHE.exists() and not FORCE:
    with open(CACHE, 'rb') as f:
        SCAN = pickle.load(f)
    if _cache_valid(SCAN):
        print(f'Loaded cache: {len(SCAN)} cubes')
        loaded = True
    else:
        print('Cache is stale (old format) — rescanning...')

if not loaded:
    SCAN = {}
    for entry in CUBES:
        zi = entry['z_info']
        if zi is None: continue
        z = zi['z']; band = entry['band']
        try:
            with fits.open(entry['path']) as h:
                hdr  = h['DATA'].header
                cube = np.array(h['DATA'].data, dtype=float)
                wave = wl_axis(hdr)
        except Exception as e:
            print(f"  skip {entry['path'].name}: {e}"); continue

        if   in_band(wave, 0.656281, z): cy,cx = peak_center(cube,wave,z,0.656281)
        elif in_band(wave, 0.500824, z): cy,cx = peak_center(cube,wave,z,0.500824)
        elif in_band(wave, 0.671644, z): cy,cx = peak_center(cube,wave,z,0.671644)
        else: cy,cx = cube.shape[1]//2, cube.shape[2]//2

        spec = ap_spec(cube, cy, cx)
        if not np.any(np.isfinite(spec)): continue

        snrs = {}
        for g in GROUPS:
            r = fit_group(wave, spec, g, z, band)
            snrs[g] = r['snr'] if r else 0.0

        SCAN[(entry['field'], entry['tid'], entry['band'])] = dict(
            entry=entry, z=z, band=band, cy=cy, cx=cx,
            wave=wave, spec=spec, snrs=snrs,
        )

    with open(CACHE, 'wb') as f: pickle.dump(SCAN, f)
    print(f'Scanned {len(SCAN)} cubes → cache saved.')

# Print top-5 per group
for g in GROUPS:
    ranked = sorted(SCAN.items(), key=lambda x: -x[1]['snrs'].get(g, 0))
    top5   = [(k,v) for k,v in ranked if v['snrs'].get(g, 0) > 1][:5]
    snr_strs = ', '.join(f"{v['snrs'][g]:.1f}" for _,v in top5)
    print(f'{g:12s}  top-5 SNR: {snr_strs}')

Loaded cache: 80 cubes
Ha+[NII]      top-5 SNR: 33.2, 21.4, 20.8, 9.6, 8.8
[OIII]        top-5 SNR: 9.2, 8.0, 6.8, 6.7, 6.7
[SII]         top-5 SNR: 17.8, 6.6, 3.3, 2.9, 2.7
Hbeta         top-5 SNR: 6.7, 5.6, 5.3, 5.2, 4.8


In [67]:
# ── Phase 2: Voronoi-binned maps for top-5 per group ──────────────────────────

try:
    from vorbin.voronoi_2d_binning import voronoi_2d_binning as _vorbin
    HAS_VORBIN = True
    print('vorbin available')
except ImportError:
    HAS_VORBIN = False
    print('vorbin not found — pip install vorbin  (using pixel-by-pixel fallback)')

TARGET_SN = 3.0   # S/N per Voronoi bin; lower = more bins, better spatial resolution


def _signal_noise_maps(cube, wave, z, band, group):
    """
    Per-spaxel noise map (MAD in continuum flanks) and peak signal near line.
    ALL spaxels with finite noise are included; signal is floored at the noise
    level so every pixel participates in Voronoi binning.
    """
    _, anchor, _ = GROUPS[group]
    lam = anchor * (1 + z)
    si  = sig_inst(band, lam)
    sig_win  = np.abs(wave - lam) < si * 2.5
    cont_win = (np.abs(wave - lam) > si * 5) & (np.abs(wave - lam) < FIT_WIN)
    ny, nx = cube.shape[1], cube.shape[2]
    S = np.full((ny, nx), np.nan)
    N = np.full((ny, nx), np.nan)
    for iy in range(ny):
        for ix in range(nx):
            ps = cube[:, iy, ix]
            if not np.any(np.isfinite(ps)): continue
            if sig_win.sum() < 2 or cont_win.sum() < 3: continue
            cm    = np.nanmedian(ps[cont_win])
            noise = max(1.4826 * np.nanmedian(np.abs(ps[cont_win] - cm)), 1e-20)
            peak  = np.nanmax(ps[sig_win]) - cm   # may be negative
            N[iy, ix] = noise
            # floor signal so every pixel with finite noise joins the binning
            S[iy, ix] = max(peak, noise)
    return S, N


def _pixel_maps(cube, wave, z, band, group, ap, cy, cx, key):
    """Plain pixel-by-pixel fitting (fallback when vorbin absent)."""
    ny, nx = cube.shape[1], cube.shape[2]
    VEL = np.full((ny, nx), np.nan)
    SIG = np.full((ny, nx), np.nan)
    SNR = np.full((ny, nx), np.nan)
    fkeys = [k for k in ap['fluxes'] if not k.startswith('ratio')]
    flux_maps = {k: np.full((ny, nx), np.nan) for k in fkeys}
    for iy in range(ny):
        for ix in range(nx):
            ps = cube[:, iy, ix]
            if not np.any(np.isfinite(ps)): continue
            r = fit_group(wave, ps, group, z, band, tight_p0=ap['popt'])
            if r is None: continue
            VEL[iy, ix] = r['vel_kms']
            SIG[iy, ix] = r['sigma_kms']
            SNR[iy, ix] = r['snr']
            for k in flux_maps:
                if k in r['fluxes']:
                    flux_maps[k][iy, ix] = r['fluxes'][k]
    spec = ap_spec(cube, cy, cx)
    return dict(ap=ap, VEL=VEL, SIG=SIG, SNR=SNR, BIN=None,
                flux_maps=flux_maps, cy=cy, cx=cx,
                wave=wave, spec=spec, z=z, band=band,
                field=key[0], tid=key[1])


def voronoi_maps(key, scan_entry, group):
    """Voronoi-bin to TARGET_SN, fit each bin's summed spectrum."""
    z    = scan_entry['z']
    band = scan_entry['band']
    cy   = scan_entry['cy']
    cx   = scan_entry['cx']
    wave = scan_entry['wave']

    with fits.open(scan_entry['entry']['path']) as h:
        cube = np.array(h['DATA'].data, dtype=float)

    ny, nx = cube.shape[1], cube.shape[2]

    spec = ap_spec(cube, cy, cx)
    ap   = fit_group(wave, spec, group, z, band)
    if ap is None:
        return None

    if not HAS_VORBIN:
        return _pixel_maps(cube, wave, z, band, group, ap, cy, cx, key)

    S, N = _signal_noise_maps(cube, wave, z, band, group)
    valid = np.isfinite(S) & np.isfinite(N) & (N > 0)

    if valid.sum() < 4:
        return _pixel_maps(cube, wave, z, band, group, ap, cy, cx, key)

    yy, xx = np.mgrid[0:ny, 0:nx]
    xv = xx[valid].astype(float)
    yv = yy[valid].astype(float)
    sv = S[valid]
    nv = N[valid]

    try:
        bin_num, *_ = _vorbin(xv, yv, sv, nv, TARGET_SN, plot=False, quiet=True)
    except Exception as e:
        print(f'  vorbin error: {e} → pixel fallback')
        return _pixel_maps(cube, wave, z, band, group, ap, cy, cx, key)

    n_bins = int(bin_num.max()) + 1
    valid_flat = np.where(valid.ravel())[0]

    bin_results = {}
    for b in range(n_bins):
        idx_in_bin = valid_flat[bin_num == b]
        spectra = []
        for flat_idx in idx_in_bin:
            iy_, ix_ = divmod(int(flat_idx), nx)
            ps = cube[:, iy_, ix_]
            if np.any(np.isfinite(ps)):
                spectra.append(ps)
        if not spectra:
            continue
        bspec = np.nansum(np.stack(spectra), axis=0)
        r = fit_group(wave, bspec, group, z, band, p0_over=ap['popt'])
        if r:
            bin_results[b] = r

    VEL = np.full((ny, nx), np.nan)
    SIG = np.full((ny, nx), np.nan)
    SNR = np.full((ny, nx), np.nan)
    BIN = np.full((ny, nx), np.nan)
    first = next(iter(bin_results.values()), None)
    fkeys = [k for k in first['fluxes'] if not k.startswith('ratio')] if first else []
    flux_maps = {k: np.full((ny, nx), np.nan) for k in fkeys}

    for i, flat_idx in enumerate(valid_flat):
        iy_, ix_ = divmod(int(flat_idx), nx)
        b = bin_num[i]
        BIN[iy_, ix_] = b
        r = bin_results.get(b)
        if r is None:
            continue
        VEL[iy_, ix_] = r['vel_kms']
        SIG[iy_, ix_] = r['sigma_kms']
        SNR[iy_, ix_] = r['snr']
        for k in flux_maps:
            if k in r['fluxes']:
                flux_maps[k][iy_, ix_] = r['fluxes'][k]

    return dict(ap=ap, VEL=VEL, SIG=SIG, SNR=SNR, BIN=BIN,
                flux_maps=flux_maps, cy=cy, cx=cx,
                wave=wave, spec=spec, z=z, band=band,
                field=key[0], tid=key[1], n_bins=n_bins)


# ── Run top-5 per group ────────────────────────────────────────────────────────
TOP5_MAPS = {}
N_TOP = 5

for group in GROUPS:
    ranked = sorted(SCAN.items(), key=lambda x: -x[1]['snrs'].get(group, 0))
    top5   = [(k, v) for k, v in ranked if v['snrs'].get(group, 0) > 1.5][:N_TOP]
    results = []
    for key, sv in top5:
        label = f"{key[0][-18:]}/{key[1]}"
        print(f'  {group:12s} {label:30s} SNR={sv["snrs"][group]:.1f} ...', end=' ')
        m = voronoi_maps(key, sv, group)
        if m:
            results.append(m)
            nb = m.get('n_bins')
            print(f"OK  ({nb} bins)" if nb else "OK  (pixel-by-pixel)")
        else:
            print('failed')
    TOP5_MAPS[group] = results
    print()

print('All maps done.')

vorbin available
  Ha+[NII]     _0.6_0.8_h21.4.cat/1168518     SNR=33.2 ... OK  (70 bins)
  Ha+[NII]     _0.6_0.8_h21.4.cat/1147707     SNR=21.4 ... OK  (126 bins)
  Ha+[NII]     cdfs_targets_2.cat/23420       SNR=20.8 ... OK  (71 bins)
  Ha+[NII]     _0.6_0.8_h21.4.cat/1226372     SNR=9.6 ... OK  (41 bins)
  Ha+[NII]     _0.6_0.8_h21.4.cat/1170582     SNR=8.8 ... OK  (57 bins)

  [OIII]       MM2215_EMPOWER.cat/8           SNR=9.2 ... OK  (80 bins)
  [OIII]       cdfs_targets_2.cat/23420       SNR=8.0 ... OK  (46 bins)
  [OIII]       MM2215_EMPOWER.cat/32650       SNR=6.8 ... OK  (133 bins)
  [OIII]       MM2215_EMPOWER.cat/29284       SNR=6.7 ... OK  (93 bins)
  [OIII]       MM2215_EMPOWER.cat/32152       SNR=6.7 ... OK  (49 bins)

  [SII]        _0.6_0.8_h21.4.cat/1226372     SNR=17.8 ... OK  (75 bins)
  [SII]        _0.6_0.8_h21.4.cat/1132608     SNR=6.6 ... OK  (188 bins)
  [SII]        _0.6_0.8_h21.4.cat/1147707     SNR=3.3 ... OK  (59 bins)
  [SII]        _0.6_0.8_h21.4.cat/1137

In [68]:
# ── Plotting helpers ──────────────────────────────────────────────────────────

def _cb(ax, im, label=''):
    div = make_axes_locatable(ax)
    cax = div.append_axes('right', size='5%', pad=0.03)
    plt.colorbar(im, cax=cax, label=label)

def _lims(arr, lo=5, hi=98, sym=False):
    v = arr[np.isfinite(arr)]
    if len(v) < 4: return 0,1
    a,b = np.percentile(v,lo), np.percentile(v,hi)
    if sym: mx=max(abs(a),abs(b)); return -mx,mx
    return a,b

def draw_map(ax, data, cmap, label, sym=False, vlo=None, vhi=None):
    if vlo is None or vhi is None:
        vlo2, vhi2 = _lims(data, sym=sym)
        vlo = vlo if vlo is not None else vlo2
        vhi = vhi if vhi is not None else vhi2
    if sym:
        mx = max(abs(vlo),abs(vhi),1)
        im = ax.imshow(data, origin='lower', interpolation='nearest',
                       cmap=cmap, vmin=-mx, vmax=mx)
    else:
        im = ax.imshow(data, origin='lower', interpolation='nearest',
                       cmap=cmap, vmin=max(vlo,0), vmax=vhi)
    _cb(ax, im, label)
    ax.tick_params(labelsize=6)

def draw_spectrum(ax, m, group):
    fn, anchor, _ = GROUPS[group]
    z = m['z']; band = m['band']
    ap = m['ap']
    ww = m['wave']; ss = m['spec']
    win = np.abs(ww - anchor*(1+z)) < FIT_WIN*1.05
    wplot = ww[win]*1e4; splot = ss[win]

    ax.step(wplot, splot, where='mid', color='0.25', lw=0.9, label='data')
    wf = np.linspace(ww[win].min(), ww[win].max(), 600)
    ax.plot(wf*1e4, fn(wf, *ap['popt']), 'r-', lw=1.3, label='model')
    ax.plot(wf*1e4, ap['popt'][0]*wf + ap['popt'][1], '--',
            color='steelblue', lw=0.8, label='cont')

    # vertical line markers
    line_rest = {
        'Ha+[NII]': [('Hα',0.656281,'#e84545'), ('[NII]6583',0.658345,'#f97306'),
                     ('[NII]6548',0.654805,'#ffa040')],
        '[OIII]':   [('[OIII]5007',0.500824,'#4fc3f7'), ('[OIII]4959',0.495940,'#81d4fa')],
        '[SII]':    [('[SII]6716',0.671644,'#66bb6a'), ('[SII]6731',0.673082,'#2d8a40')],
        'Hbeta':    [('Hβ',0.486270,'#ce93d8')],
    }
    yhi = np.nanmax(splot[np.isfinite(splot)]) if np.any(np.isfinite(splot)) else 1
    for lname, lr, lc in line_rest.get(group,[]):
        lobs = lr*(1+ap['z_fit'])*1e4
        ax.axvline(lobs, ls=':', lw=0.8, color=lc, alpha=0.9)
        ax.text(lobs, yhi*0.88, lname, rotation=90, fontsize=5,
                color=lc, ha='center', va='top')

    ax.set_xlabel('λ_obs [Å]', fontsize=7)
    ax.set_ylabel('Flux', fontsize=7)
    ax.tick_params(labelsize=6)
    ax.legend(fontsize=5, loc='upper left')
    ax.set_title(f"SNR={ap['snr']:.1f}  v={ap['vel_kms']:.0f} km/s"
                 f"  σ={ap['sigma_kms']:.0f} km/s", fontsize=7)

print('Plot helpers ready')

Plot helpers ready


In [ ]:
# ── Display: one figure per line group, one row per galaxy ────────────────────
# Columns: flux(es) | velocity [Spectral_r centred at 0 for Hα] | σ_int | spectrum

VEL_CMAP = {
    'Ha+[NII]': 'Spectral_r',
    '[OIII]':   'RdBu_r',
    '[SII]':    'RdBu_r',
    'Hbeta':    'RdBu_r',
}

for group, results in TOP5_MAPS.items():
    if not results:
        print(f'No results for {group}'); continue

    sample_flux_keys = list(results[0]['flux_maps'])
    n_flux  = len(sample_flux_keys)
    n_cols  = n_flux + 3          # flux(es) + vel + sigma + spectrum
    nrows   = len(results)
    vcmap   = VEL_CMAP.get(group, 'RdBu_r')

    fig, axes = plt.subplots(nrows, n_cols,
                             figsize=(n_cols * 2.8, nrows * 3.0),
                             squeeze=False)
    fig.suptitle(f'{group}  —  top {nrows} by aperture SNR  '
                 f'(Voronoi S/N≥{TARGET_SN})',
                 fontsize=11, fontweight='bold', y=1.01)

    flux_cmaps = ['inferno', 'magma', 'cividis']

    for row, m in enumerate(results):
        tid_short = m['tid'][-10:]
        nb_str = f"{m['n_bins']}b" if m.get('n_bins') else 'pp'
        axes[row, 0].set_ylabel(
            f"{tid_short}\nz={m['z']:.4f} {m['band']}\n[{nb_str}]", fontsize=7)

        col = 0

        # ── Flux maps ─────────────────────────────────────────────────────────
        for fi, fk in enumerate(sample_flux_keys):
            fmap = m['flux_maps'].get(fk, np.full((14, 14), np.nan))
            draw_map(axes[row, col], fmap, flux_cmaps[fi % len(flux_cmaps)], label=fk)
            axes[row, col].plot(m['cx'], m['cy'], '+', color='lime',
                                ms=7, mew=1.5, zorder=10)
            if row == 0:
                axes[row, col].set_title(fk, fontsize=8)
            col += 1

        # ── Velocity map ──────────────────────────────────────────────────────
        vel_data = m['VEL']
        fin = vel_data[np.isfinite(vel_data)]
        mx_v = max(np.percentile(np.abs(fin), 95), 20.) if len(fin) > 3 else 100.
        im_v = axes[row, col].imshow(vel_data, origin='lower',
                                      interpolation='nearest',
                                      cmap=vcmap, vmin=-mx_v, vmax=mx_v)
        _cb(axes[row, col], im_v, 'km/s')
        axes[row, col].tick_params(labelsize=6)
        axes[row, col].plot(m['cx'], m['cy'], '+', color='k',
                            ms=7, mew=1.5, zorder=10)
        if row == 0:
            axes[row, col].set_title(f'velocity', fontsize=8)
        col += 1

        # ── Sigma map ─────────────────────────────────────────────────────────
        draw_map(axes[row, col], m['SIG'], 'plasma', 'km/s')
        axes[row, col].plot(m['cx'], m['cy'], '+', color='lime',
                            ms=7, mew=1.5, zorder=10)
        if row == 0:
            axes[row, col].set_title('σ_int', fontsize=8)
        col += 1

        # ── Spectrum + model ───────────────────────────────────────────────────
        draw_spectrum(axes[row, col], m, group)
        if row == 0:
            axes[row, col].set_title('aperture spectrum + fit', fontsize=8)

    plt.tight_layout()
    plt.show()